# Exploring the LGN atlas live

`dynaphos_lgn.viz3d` and `dynaphos_lgn.slices` write **figures**: a
self-contained HTML file you can email, in which every state a control
can reach was computed in advance. That is what makes such a file
shareable, and it is also what caps how interactive it can be. A slider
there is a lookup table, so an arbitrary slice position, a different
spacing between slices, or a switch from coronal to sagittal is not
something the file can do &mdash; it would have to contain every one of
them.

This notebook is the other half. Same volume, same overlays, same
colours, same electrodes, driven from `ipywidgets` with the kernel
still attached. Nothing is precomputed, so the controls are free.

The price is that this produces no file. That is what the last section
is for: the explorer keeps its state in the parameter dictionary rather
than in the widgets, so at any moment it can hand back the YAML that
reproduces what you are looking at, and the `slices.py` command line
that renders it as a static figure.

## 0. Requirements

Beyond the package's own dependencies this needs `ipywidgets`, `plotly`
and `anywidget` (which is what plotly's `FigureWidget` is built on):

```
pip install ipywidgets plotly anywidget
```

`python -m dynaphos_lgn.explore` checks for them from a terminal and
says what is missing.

In [1]:
import sys
sys.path.append('./..')          # running from examples/

from dynaphos_lgn import explore

ATLAS_DIR = '../data/Erwin_Atlas'

found, missing = explore.check_environment()
print('found  :', ', '.join(found))
print('missing:', ', '.join(missing) or '(none)')

found  : ipywidgets, plotly, anywidget
missing: (none)


## 1. Open the explorer

`Explorer.open` takes the same steps `python -m dynaphos_lgn.slices`
does, in the same order and through the same builders, so what you see
here is what that command would draw.

Loading is the slow part &mdash; the atlas, the shell mesh and a
volume-wide colour sample &mdash; and it happens once. Everything the
controls then do is one plane's worth of work.

Set `synthetic=True` to run this on the stand-in atlas with no download:
it exercises every control and means nothing biologically.

In [2]:
ex = explore.Explorer.open(
    atlas_dir=ATLAS_DIR,
    hemispheres=['left'],        # add 'right' for both nuclei
    overlays=['layer', 'eccentricity', 'cell_class', 'eye'],
    electrodes='layout',         # 'none', 'layout' or 'array'
    # synthetic=True,            # no atlas download needed
)
print(ex.volume.describe())

C:\Users\sebas\PycharmProjects\dynaphos_LGN\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


LGN explorer volume: Erwin, Baker, Busen & Malpeli (1999) macaque LGN atlas
  tissue shown      : all reconstructed tissue
  scene grid        : 219 x 258 x 290 voxels of 25 um (left LGN)
  overlays offered  : layer, eccentricity, cell_class, eye
  shell             : 19,256 faces at stride 4
  electrodes        : 64 (64 electrodes, farthest-point sampled in the visual field (left LGN))


## 2. The viewer

On the left the nucleus as a see-through shell, with the cut planes
drawn where they actually cut. On the right the plane the position
control is on, flat, one image pixel per atlas voxel.

The controls, top to bottom:

| control | what it does |
|---|---|
| **Overlay** | which per-voxel quantity is painted on the planes |
| **Plane** | coronal, sagittal or horizontal. Each remembers where you last were on it |
| **AP / DV / ML (mm)** | where the cut is, one 25 µm voxel at a time. This is the position control; it is named after the axis it moves along, so its label changes with the plane |
| **Planes** | how many parallel cuts to draw at once |
| **Spacing (mm)** | how far apart they sit |
| **Slab (mm)** | how close a contact must be to a plane to be drawn on it |
| **Shell** | the nucleus's opacity |
| **Plane stride** | every nth voxel of the in-scene planes &mdash; coarser is faster |

The plane the flat panel is showing is opaque; the rest of the stack is
not, so you can tell which one you are reading.

The line under the figure reports what was actually drawn rather than
what was asked for. Three things can legitimately differ: the spacing
snaps to whole 25 µm voxels, a stack near the end of the nucleus loses
the members that fall outside it, and a position can land where there
is no tissue at all.

It also reports how long the redraw took, which is worth watching. A
plane and a panel are two-dimensional arrays, and plotly's widget ships
only one-dimensional ones to the browser as binary, so every redraw
sends a few megabytes of JSON. One nucleus is about 100 ms; both, with
a stack of nine, nearer 350 ms, because a coronal panel spanning both
then has the real 17 mm midline gap in the middle of it. If it drags,
the knobs are **Planes**, **Plane stride**, and opening one nucleus
rather than two. `visualization.explorer.continuous_update` decides
whether a slider redraws as you drag it or when you let go; it lets go
by default, for the same reason.

In [3]:
ex.widget

### Driving it from code

The controls are ordinary widgets, so a cell can set them and the
figure follows. Useful for getting back to a view you described in
prose, and for scripting a series of positions.

In [4]:
ex.plane_control.value = 'coronal'
ex.overlay_control.value = 'layer'
ex.count_control.value = 5
ex.spacing_control.value = 0.4
ex.position_control.value = explore.fullest_plane_mm(
    ex.volume, ex.view().stack.plane)

print(ex.status.value.replace('&middot;', '|').replace('&mdash;', '--'))

19,101 voxels in this coronal plane | 5 planes at 0.400 mm | 15 contact(s) within 250 um | redrawn in 461 ms


## 3. Changing which tissue is shown

Cell class, laminae and eccentricity are under **Tissue selection** in
the accordion. These are different in kind from the controls above:
each changes which voxels exist, so applying one rebuilds the masks,
the scene grid, the shell mesh and the volume-wide colour sample. That
is seconds rather than milliseconds, which is why it is a button and
not something that fires as you drag.

The same from code:

In [5]:
ex.cell_class_control.value = 'magno'
ex.apply_selection()
print(ex.volume.selection)
print(ex.volume.describe())

magno laminae only
LGN explorer volume: Erwin, Baker, Busen & Malpeli (1999) macaque LGN atlas
  tissue shown      : magno laminae only
  scene grid        : 142 x 215 x 255 voxels of 25 um (left LGN)
  overlays offered  : layer, eccentricity, cell_class, eye
  shell             : 10,298 faces at stride 4
  electrodes        : 64 (64 electrodes, farthest-point sampled in the visual field (left LGN))


In [6]:
ex.cell_class_control.value = None      # back to all of it
ex.apply_selection()

## 4. Exporting the settings

The explorer writes every control's value into `ex.params` at the
dotted path the rest of the package reads it from. Exporting is
therefore a diff against `config/params_lgn.yaml` rather than a second
inventory of what the controls happen to say &mdash; the config keeps
being the one place a number lives.

You get two things, and they are not alternatives:

- a **YAML fragment** of everything that differs from the file on disk,
  nested so it can be merged back into `config/params_lgn.yaml` by
  hand. This is the durable record.
- a **command line** that renders the same picture as a static figure.
  It restates as flags everything that has one, so in the ordinary case
  it needs nothing else.

The two never disagree: each flag folds into exactly the config key the
explorer wrote it to.

In [7]:
print(ex.export().text)

# LGN explorer settings.
#
# The same picture as a static figure:
#   python -m dynaphos_lgn.slices --view combined --plane coronal --atlas-dir C:\Users\sebas\PycharmProjects\dynaphos_LGN\data\Erwin_Atlas --overlay layer eccentricity cell_class eye --count 5 --range 6.7500:8.3500 --electrodes layout
#
# Merge the block below into config/params_lgn.yaml.

visualization:
  overlays:
  - layer
  - eccentricity
  - cell_class
  - eye
  slices:
    count: 5
    range_mm:
    - 6.75
    - 8.35
    min_slice_fraction: 0.0
  combined:
    min_slice_fraction: 0.0
  explorer:
    count: 5
    spacing_mm: 0.4


### Writing it out

`save()` writes the fragment, under `visualization.output.directory`
and the name in `visualization.explorer.export`. The command goes in a
comment, so the `.yaml` file really is YAML.

`save(full=True)` writes the complete parameter dictionary instead,
giving a file `--params` can be pointed at. It loses every comment in
`params_lgn.yaml`, which in this package carry most of the reasoning,
so it is for feeding back to the program rather than for reading.

In [8]:
exported = ex.export()
print(exported.save())
print(exported.save(full=True))

C:\Users\sebas\PycharmProjects\dynaphos_LGN\examples\out\lgn_explorer_settings.yaml
C:\Users\sebas\PycharmProjects\dynaphos_LGN\examples\out\params_lgn_explorer.yaml


## 5. The same picture as a static figure

Which is the point of the export: explore here, then write a file you
can put in a paper or hand to someone with no Python.

In [9]:
from dynaphos_lgn import slices

exported = ex.export()
print(exported.command)          # what you would type in a terminal

# `tokens` is the same arguments, unquoted and ready for Python. Use it
# rather than splitting `command` back up: shlex.split is POSIX, so on
# Windows it would eat the backslashes in an atlas path.
slices.main(exported.tokens)

python -m dynaphos_lgn.slices --view combined --plane coronal --atlas-dir C:\Users\sebas\PycharmProjects\dynaphos_LGN\data\Erwin_Atlas --overlay layer eccentricity cell_class eye --count 5 --range 6.7500:8.3500 --electrodes layout
LGN slices: Erwin, Baker, Busen & Malpeli (1999) macaque LGN atlas
  tissue drawn      : all reconstructed tissue
  scene grid        : 219 x 258 x 290 voxels of 25 um (left LGN)
  coronal    AP = +6.750 mm : 17,300 voxels of 258 x 219
  coronal    AP = +7.150 mm : 17,768 voxels of 258 x 219
  coronal    AP = +7.550 mm : 19,101 voxels of 258 x 219
  coronal    AP = +7.950 mm : 18,309 voxels of 258 x 219
  coronal    AP = +8.350 mm : 17,051 voxels of 258 x 219
  overlays          : layer, eccentricity, cell_class, eye
    layer                      categorical
    eccentricity               0 to 96.2
    cell_class                 categorical
    eye                        categorical
  shell             : 19,256 faces at stride 4, around the slices
  electrod

0

---

### Where to go next

- `python -m dynaphos_lgn.viz3d` &mdash; the nucleus as a surface with
  an overlay painted on it, which is the one thing this notebook does
  not do: the shell here is deliberately one flat colour, because two
  coloured surfaces in one scene fight each other.
- `python -m dynaphos_lgn.slices --view ortho` &mdash; coronal,
  sagittal and horizontal through one point, with a crosshair on each
  showing where the other two cut.
- `python -m dynaphos_lgn.slices --view combined` &mdash; the static
  counterpart of this viewer.
- `dynaphos_lgn/README.md` &mdash; what every overlay means, and which
  of them are derived rather than measured.